# Week 4 Task 4 — Reproducibility of a Full Platform Rebuild

The `B_full_rebuild` timing experiment rebuilt every platform table four times in a separate copy of the project (`~/task4_rebuild_copy`), starting from the same raw files. This notebook asks whether rebuilding the platform reproduces the same data.

Two comparisons answer two different questions:

| Comparison | Question |
|---|---|
| **Run to run**: first rebuild (Delta version 0) vs last rebuild (latest version), both in the copy | Does the same code, environment, and input always produce the same table? |
| **Original vs rebuild**: tables in this repository vs the latest rebuild in the copy | Does a rebuild today reproduce the tables the team built earlier in the course? |

Only **business columns** are compared. Columns starting with `_` are platform audit metadata (`_source_file`, `_ingestion_run_id`, `_ingested_at`, `_integrated_at`, …). They change on every run by design, and `_source_file` also contains the absolute path of the project.

Each comparison counts rows that appear in one table but not the other with `exceptAll`, which treats the tables as multisets: duplicates must also match.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import sys

import pandas as pd
from delta.tables import DeltaTable


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
COPY_ROOT = Path(os.environ.get("TASK4_REBUILD_COPY", Path.home() / "task4_rebuild_copy")).resolve()
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "task4_reproducibility.json"
assert COPY_ROOT != PROJECT_ROOT and COPY_ROOT.is_dir(), f"Rebuild copy not found: {COPY_ROOT}"

source_package = str(PROJECT_ROOT / "src")
if source_package not in sys.path:
    sys.path.insert(0, source_package)

from data_intensive_computing.spark import build_spark_session

spark = build_spark_session(
    "Week4-Task4-Reproducibility",
    driver_memory="8g",
    shuffle_partitions=32,
)

TABLES = {
    "taxi_zones": "data/processed/taxi_zones",
    "weather": "data/processed/weather",
    "taxi_trips": "data/processed/taxi_trips",
    "air_quality": "data/processed/air_quality",
    "integrated_taxi_trips": "data/processed/integrated_taxi_trips",
    "fare_training_dataset": "data/processed/ml/fare_training_dataset",
}

print("Original project:", PROJECT_ROOT)
print("Rebuild copy:    ", COPY_ROOT)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ba142648-023a-4ed0-a6df-504ce469f80e;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Original project: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing
Rebuild copy:     /Users/rexchang/task4_rebuild_copy


## 1. Rebuild history in the copy

Every rebuild overwrites each table, so the copy should hold one Delta version per rebuild: versions 0 to 3 for the warm-up and three measured runs.

In [2]:
history_rows = []
for name, relative in TABLES.items():
    history = DeltaTable.forPath(spark, str(COPY_ROOT / relative)).history().select("version", "operation").collect()
    versions = sorted(row["version"] for row in history)
    history_rows.append(
        {
            "table": name,
            "versions": f"{versions[0]}–{versions[-1]}",
            "operations": sorted({row["operation"] for row in history}),
        }
    )
display(pd.DataFrame(history_rows))

,table,versions,operations
0,taxi_zones,0–3,[WRITE]
1,weather,0–3,[WRITE]
2,taxi_trips,0–3,[WRITE]
3,air_quality,0–3,[WRITE]
4,integrated_taxi_trips,0–3,[WRITE]
5,fare_training_dataset,0–3,[WRITE]


## 2. Comparison helper

In [3]:
def business_columns(df):
    return [c for c in df.columns if not c.startswith("_")]


def compare_tables(left, right) -> dict:
    left_cols, right_cols = business_columns(left), business_columns(right)
    result = {
        "same_business_schema": [(f.name, f.dataType.simpleString()) for f in left.select(left_cols).schema.fields]
        == [(f.name, f.dataType.simpleString()) for f in right.select(right_cols).schema.fields],
    }
    if not result["same_business_schema"]:
        result["only_left_columns"] = sorted(set(left_cols) - set(right_cols))
        result["only_right_columns"] = sorted(set(right_cols) - set(left_cols))
        common = [c for c in left_cols if c in set(right_cols)]
    else:
        common = left_cols

    left_rows = left.select(common).cache()
    right_rows = right.select(common).cache()
    result["left_rows"] = left_rows.count()
    result["right_rows"] = right_rows.count()
    result["only_left_rows"] = left_rows.exceptAll(right_rows).count()
    result["only_right_rows"] = right_rows.exceptAll(left_rows).count()
    result["identical"] = (
        result["same_business_schema"] and result["only_left_rows"] == 0 and result["only_right_rows"] == 0
    )
    left_rows.unpersist()
    right_rows.unpersist()
    return result


def read_delta(root: Path, relative: str, version: int | None = None):
    reader = spark.read.format("delta")
    if version is not None:
        reader = reader.option("versionAsOf", version)
    return reader.load(str(root / relative))

## 3. Run to run: first vs last rebuild in the copy

Same code, same Spark version, same raw files, same machine. Any difference here is non-determinism inside the platform itself, for example deduplication keeping a different row when two duplicates tie on the sort order.

In [4]:
run_to_run = {}
for name, relative in TABLES.items():
    latest = DeltaTable.forPath(spark, str(COPY_ROOT / relative)).history(1).first()["version"]
    run_to_run[name] = {
        "left": "copy@v0",
        "right": f"copy@v{latest}",
        **compare_tables(read_delta(COPY_ROOT, relative, 0), read_delta(COPY_ROOT, relative)),
    }
    print(f"{name:24s} identical={run_to_run[name]['identical']}")

display(pd.DataFrame(run_to_run).T)

26/10/04 13:39:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


taxi_zones               identical=True
weather                  identical=True


taxi_trips               identical=True


air_quality              identical=True


26/10/04 13:41:01 WARN MemoryStore: Not enough space to cache rdd_567_20 in memory! (computed 167.0 MiB so far)
26/10/04 13:41:01 WARN MemoryStore: Not enough space to cache rdd_567_26 in memory! (computed 35.5 MiB so far)


integrated_taxi_trips    identical=True


fare_training_dataset    identical=True


,left,right,same_business_schema,left_rows,right_rows,only_left_rows,only_right_rows,identical
taxi_zones,copy@v0,copy@v3,True,265,265,0,0,True
weather,copy@v0,copy@v3,True,8783,8783,0,0,True
taxi_trips,copy@v0,copy@v3,True,9417689,9417689,0,0,True
air_quality,copy@v0,copy@v3,True,7927367,7927367,0,0,True
integrated_taxi_trips,copy@v0,copy@v3,True,9417689,9417689,0,0,True
fare_training_dataset,copy@v0,copy@v3,True,9380457,9380457,0,0,True


## 4. Original vs rebuild

The original tables in this repository were built earlier in the course and have since gone through the Week 3 incremental-update and restore experiments. A difference here can come from data that was not restored, from code changes since the original build, or from a different Spark or Delta version.

In [5]:
original_vs_rebuild = {}
for name, relative in TABLES.items():
    original_vs_rebuild[name] = {
        "left": "original",
        "right": "copy@latest",
        **compare_tables(read_delta(PROJECT_ROOT, relative), read_delta(COPY_ROOT, relative)),
    }
    print(f"{name:24s} identical={original_vs_rebuild[name]['identical']}")

display(pd.DataFrame(original_vs_rebuild).T)

taxi_zones               identical=True
weather                  identical=True


taxi_trips               identical=True


air_quality              identical=True


26/10/04 13:42:41 WARN MemoryStore: Not enough space to cache rdd_1167_18 in memory! (computed 167.0 MiB so far)
26/10/04 13:42:41 WARN MemoryStore: Not enough space to cache rdd_1167_20 in memory! (computed 167.0 MiB so far)


integrated_taxi_trips    identical=True


fare_training_dataset    identical=True


,left,right,same_business_schema,left_rows,right_rows,only_left_rows,only_right_rows,identical
taxi_zones,original,copy@latest,True,265,265,0,0,True
weather,original,copy@latest,True,8783,8783,0,0,True
taxi_trips,original,copy@latest,True,9417689,9417689,0,0,True
air_quality,original,copy@latest,True,7927367,7927367,0,0,True
integrated_taxi_trips,original,copy@latest,True,9417689,9417689,0,0,True
fare_training_dataset,original,copy@latest,True,9380457,9380457,0,0,True


## 5. Save the results

In [6]:
results = {
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "original_root": str(PROJECT_ROOT),
    "rebuild_copy_root": str(COPY_ROOT),
    "spark_version": spark.version,
    "compared_columns": "business columns only (columns not starting with '_')",
    "run_to_run": run_to_run,
    "original_vs_rebuild": original_vs_rebuild,
}
RESULTS_PATH.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
print("Results written to", RESULTS_PATH)

Results written to /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/artifacts/task4_reproducibility.json
